# Diabetes Risk Prediction — Complete Pipeline

**Dataset:** Diabetes Health Indicators (~253k rows, 22 columns)  
**Goal:** EDA → Cleaning → Feature Engineering → Model Training & Comparison

---
## Pipeline Sections
1. Setup & Imports
2. EDA — shape, types, distributions, correlations
3. Data Cleaning — nulls, duplicates, outliers
4. Feature Engineering
5. Model Training — Logistic Regression, Decision Tree, Random Forest, XGBoost
6. Results Comparison

---
## 1. Setup & Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
import mlflow
import mlflow.sklearn

pd.set_option('display.max_columns', None)

In [ ]:
# Load raw dataset
df_diabetes = pd.read_csv('../data/diabetes.csv')
print(f'Raw dataset shape: {df_diabetes.shape}')

---
## 2. EDA

In [ ]:
print('Shape:', df_diabetes.shape, '\n')

df_diabetes.info()

In [ ]:
# Summary statistics
df_diabetes.describe().T

In [ ]:
# Correlation heatmap (with correlation values)
corr = df_diabetes.corr(numeric_only=True)

plt.figure(figsize=(16, 12))
sns.heatmap(
    corr,
    cmap='coolwarm',
    center=0,
    annot=True,
    fmt='.2f',
    annot_kws={'size': 7},
    square=True,
    linewidths=0.3,
    cbar_kws={'shrink': 0.8}
 )
plt.title('Feature Correlation Heatmap', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Class distribution
target_counts = df_diabetes["Diabetes_012"].value_counts().sort_index()
target_pct = (target_counts / len(df_diabetes) * 100).round(2)

target_summary = pd.DataFrame({
    "count": target_counts,
    "percentage": target_pct
})
target_summary

In [ ]:
ax = sns.countplot(
    data=df_diabetes,
    x="Diabetes_012",
    hue="Diabetes_012",
    palette="Set2",
    legend=False
)
ax.set_title("Class Distribution of Diabetes_012")
ax.set_xlabel("Diabetes Class")
ax.set_ylabel("Count")
for container in ax.containers:
    ax.bar_label(container)
plt.show()

In [ ]:
# Scatter plot matrix (pairplot)
pairplot_cols = ['BMI', 'Age', 'GenHlth', 'PhysHlth', 'MentHlth', 'Diabetes_012']

# Sample rows for faster rendering while preserving class proportions
df_pair = df_diabetes[pairplot_cols].sample(n=3000, random_state=42)

sns.pairplot(
    data=df_pair,
    vars=['BMI', 'Age', 'GenHlth', 'PhysHlth', 'MentHlth'],
    hue='Diabetes_012',
    diag_kind='kde',
    corner=True,
    plot_kws={'alpha': 0.35, 's': 12},
    diag_kws={'fill': True},
    palette='Set2'
 )
plt.suptitle('Scatter Plot Matrix of Key Health Features', y=1.02)
plt.show()

---
## 3. Data Cleaning

In [ ]:
# Missing values
print('Missing values per column:', df_diabetes.isnull().sum())

In [ ]:
# Duplicates
n_dupes = df_diabetes.duplicated().sum()
print(f'Duplicate rows: {n_dupes:,}  ({n_dupes/len(df_diabetes)*100:.2f}%)')

# Drop duplicates — same person answering twice adds no information
df_clean = df_diabetes.drop_duplicates().reset_index(drop=True)
print(f'Shape after removing duplicates: {df_clean.shape}')

---
## 4. Feature Engineering

Most features are already binary or ordinal — minimal encoding needed.  
We'll add three meaningful derived features.

In [ ]:
# BMI category
def bmi_category(bmi):
    if bmi < 18.5: return 0   # Underweight
    if bmi < 25:   return 1   # Normal
    if bmi < 30:   return 2   # Overweight
    return 3                  # Obese

df_clean['BMI_cat'] = df_clean['BMI'].apply(bmi_category)
print('BMI category distribution:')
print(df_clean['BMI_cat'].value_counts().sort_index().rename({0:'Underweight',1:'Normal',2:'Overweight',3:'Obese'}))

In [ ]:
# Create binary target column
df_clean['Diabetes_Binary'] = (df_clean['Diabetes_012'] > 0).astype(int)
df_clean = df_clean.drop(columns=['Diabetes_012'])

print('Final dataset shape:', df_clean.shape)
print('Target balance:')
print(df_clean['Diabetes_Binary'].value_counts(normalize=True).round(3))
df_clean.head()

---
## 5. Model Training & Comparison

In [ ]:
# Prepare data for modeling
X = df_clean.drop(columns=['Diabetes_Binary'])  # features
y = df_clean['Diabetes_Binary']   # target variable

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=12, stratify=y
)

print(f'Training set size: {X_train.shape[0]}')
print(f'Test set size: {X_test.shape[0]}')

In [ ]:
# Model hyperparameters 

lr_params = {
    "model_type":    "logistic_regression",
    "C":             1.0,        # inverse regularisation strength
    "max_iter":      1000,
    "class_weight":  "balanced", # compensates for imbalance
    "random_state":  12,
}

dt_params = {
    "model_type": "decision_tree",
    "max_depth": 8,
    "min_samples_split": 20,
    "min_samples_leaf": 10,
    "class_weight": "balanced",
    "random_state": 12
}

rf_params = {
    "model_type": "random_forest",
    "n_estimators": 300,
    "max_depth": 12,
    "min_samples_split": 20,
    "min_samples_leaf": 8,
    "class_weight": "balanced_subsample",
    "random_state": 12
}

xgb_params = {
    "model_type": "xgboost",
    "n_estimators": 300,
    "max_depth": 6,
    "learning_rate": 0.1,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "scale_pos_weight": 5,
    "random_state": 12
}

In [ ]:
# MLflow experiment

mlflow.set_experiment("diabetes-risk")

In [ ]:
# Logistic Regression
with mlflow.start_run(run_name='logistic_regression'):

    model_lr = LogisticRegression(
        C=lr_params["C"],
        max_iter=lr_params["max_iter"],
        class_weight=lr_params["class_weight"],
        random_state=lr_params["random_state"],
    )
    model_lr.fit(X_train, y_train)

    y_pred_lr = model_lr.predict(X_test)
    y_proba_lr = model_lr.predict_proba(X_test)[:, 1]

    lr_metrics = {
        "accuracy": accuracy_score(y_test, y_pred_lr),
        "f1":       f1_score(y_test, y_pred_lr),
        "roc_auc":  roc_auc_score(y_test, y_proba_lr),
    }

    mlflow.log_params(lr_params)
    mlflow.log_metrics(lr_metrics)
    mlflow.sklearn.log_model(model_lr, "model")

print("Logistic Regression")
print(f"  Accuracy : {lr_metrics['accuracy']:.4f}")
print(f"  F1 Score : {lr_metrics['f1']:.4f}")
print(f"  ROC AUC  : {lr_metrics['roc_auc']:.4f}")

In [ ]:
# Decision Tree
with mlflow.start_run(run_name="decision_tree"):
    model_dt = DecisionTreeClassifier(
        max_depth=dt_params["max_depth"],
        min_samples_split=dt_params["min_samples_split"],
        min_samples_leaf=dt_params["min_samples_leaf"],
        class_weight=dt_params["class_weight"],
        random_state=dt_params["random_state"]
    )
    model_dt.fit(X_train, y_train)

    y_pred_dt = model_dt.predict(X_test)
    y_proba_dt = model_dt.predict_proba(X_test)[:, 1]

    dt_metrics = {
        "accuracy": accuracy_score(y_test, y_pred_dt),
        "f1": f1_score(y_test, y_pred_dt),
        "roc_auc": roc_auc_score(y_test, y_proba_dt),
    }

    mlflow.log_params(dt_params)
    mlflow.log_metrics(dt_metrics)
    mlflow.sklearn.log_model(model_dt, "model")

print("Decision Tree")
print(f"  Accuracy : {dt_metrics['accuracy']:.4f}")
print(f"  F1 Score : {dt_metrics['f1']:.4f}")
print(f"  ROC AUC  : {dt_metrics['roc_auc']:.4f}")

In [ ]:
# Random Forest
with mlflow.start_run(run_name="random_forest"):
    model_rf = RandomForestClassifier(
        n_estimators=rf_params["n_estimators"],
        max_depth=rf_params["max_depth"],
        min_samples_split=rf_params["min_samples_split"],
        min_samples_leaf=rf_params["min_samples_leaf"],
        class_weight=rf_params["class_weight"],
        random_state=rf_params["random_state"],
        n_jobs=-1
    )
    model_rf.fit(X_train, y_train)

    y_pred_rf = model_rf.predict(X_test)
    y_proba_rf = model_rf.predict_proba(X_test)[:, 1]

    rf_metrics = {
        "accuracy": accuracy_score(y_test, y_pred_rf),
        "f1": f1_score(y_test, y_pred_rf),
        "roc_auc": roc_auc_score(y_test, y_proba_rf),
    }

    mlflow.log_params(rf_params)
    mlflow.log_metrics(rf_metrics)
    mlflow.sklearn.log_model(model_rf, "model")

print("Random Forest")
print(f"  Accuracy : {rf_metrics['accuracy']:.4f}")
print(f"  F1 Score : {rf_metrics['f1']:.4f}")
print(f"  ROC AUC  : {rf_metrics['roc_auc']:.4f}")

In [ ]:
# XGBoost
with mlflow.start_run(run_name="xgboost"):
    model_xgb = XGBClassifier(
        n_estimators=xgb_params["n_estimators"],
        max_depth=xgb_params["max_depth"],
        learning_rate=xgb_params["learning_rate"],
        subsample=xgb_params["subsample"],
        colsample_bytree=xgb_params["colsample_bytree"],
        scale_pos_weight=xgb_params["scale_pos_weight"],
        random_state=xgb_params["random_state"],
        eval_metric="logloss"
    )
    model_xgb.fit(X_train, y_train)

    y_pred_xgb = model_xgb.predict(X_test)
    y_proba_xgb = model_xgb.predict_proba(X_test)[:, 1]

    xgb_metrics = {
        "accuracy": accuracy_score(y_test, y_pred_xgb),
        "f1": f1_score(y_test, y_pred_xgb),
        "roc_auc": roc_auc_score(y_test, y_proba_xgb),
    }

    mlflow.log_params(xgb_params)
    mlflow.log_metrics(xgb_metrics)
    mlflow.sklearn.log_model(model_xgb, "model")

print("XGBoost")
print(f"  Accuracy : {xgb_metrics['accuracy']:.4f}")
print(f"  F1 Score : {xgb_metrics['f1']:.4f}")
print(f"  ROC AUC  : {xgb_metrics['roc_auc']:.4f}")

---
## 6. Results Comparison

In [ ]:
# Create comparison table
results = pd.DataFrame({
    'Model': ['Logistic Regression', 'Decision Tree', 'Random Forest', 'XGBoost'],
    'Accuracy': [lr_metrics['accuracy'], dt_metrics['accuracy'], rf_metrics['accuracy'], xgb_metrics['accuracy']],
    'F1 Score': [lr_metrics['f1'], dt_metrics['f1'], rf_metrics['f1'], xgb_metrics['f1']],
    'ROC AUC': [lr_metrics['roc_auc'], dt_metrics['roc_auc'], rf_metrics['roc_auc'], xgb_metrics['roc_auc']]
})

results_sorted = results.sort_values('ROC AUC', ascending=False).reset_index(drop=True)
print('\n=== Model Comparison (sorted by ROC AUC) ===')
print(results_sorted.to_string(index=False))

print(f'\nBest model: {results_sorted.iloc[0]["Model"]} (ROC AUC: {results_sorted.iloc[0]["ROC AUC"]:.4f})')